# Build the labelled dataset

Research only: read a human annotation workbook, raw stills, and FlyTracker output,
then write one aligned crop per labelled fly for supervised training and evaluation.
Routine scoring uses the command line workflow in the repository README.

Run this notebook from `notebooks/`. Put the workbook at `../data/annotations.xlsx`
and recording folders under `../data/sessions/`; these inputs are not included in Git.
The workbook convention is M = male, F = female, ? = unknown, and a blank fly cell
means female only within a row containing at least one annotation. Confirm that
convention before using a different annotator's workbook.

Run again when human annotations are added. Predictions never select labels or a
fixed number of males from this workbook.


## Imports and configuration

In [ ]:
# Import dependencies
import collections
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from openpyxl import load_workbook
from scipy import ndimage
from scipy.io import loadmat

In [ ]:
# Define file locations
WORKBOOK_PATH = Path("../data/annotations.xlsx")
SESSION_ROOT = Path("../data/sessions")
OUTPUT_PATH = Path("../data/labeled_crops.npz")

# Map workbook tab names to session folder names
SESSION_FOLDERS = {
    "0714": "07142026",
    "0717": "07172026",
    "0721": "7212026",
}

# Define well crop rectangles in panorama coordinates
WELL_ORIGINS = {
    1: (260, 171),
    2: (832, 163),
    3: (1402, 171),
    4: (267, 748),
    5: (824, 748),
    6: (1387, 733),
}

# Define image geometry
PANORAMA_WIDTH = 1944
STILL_WIDTH = 2592
FRAME_OFFSET = 1
MINIMUM_FILE_SIZE = 1_500_000

# Define crop geometry
CROP_SIZE = 96
REFERENCE_PPM = 15.7
FIELD_OF_VIEW = 0.75
BACKGROUND_WINDOW = 121
INTENSITY_FLOOR = 0.35
INTENSITY_RANGE = 0.80

## Function definitions

In [ ]:
# Read the annotation workbook into a list of labelled rows
def read_annotations(workbook_path):
    # Validate the workbook path
    if not workbook_path.is_file():
        raise ValueError(f"Annotation workbook not found: {workbook_path}")

    # Open the workbook with formula results
    workbook = load_workbook(workbook_path, data_only=True)

    # Collect one record per labelled image
    records = []
    for sheet_name in workbook.sheetnames:
        # Skip sheets that are not per-well tabs
        if "_a" not in sheet_name:
            continue

        # Scan every row for an image filename
        sheet = workbook[sheet_name]
        for row_index in range(1, sheet.max_row + 1):
            filename = str(sheet.cell(row=row_index, column=1).value or "")
            if ".jpg" not in filename:
                continue

            # Read the five fly columns
            entries = [
                str(sheet.cell(row=row_index, column=column).value or "").strip().upper()
                for column in range(2, 7)
            ]

            # Skip rows with no annotation at all
            if not any(entry in ("M", "F", "?") for entry in entries):
                continue

            # Treat a blank cell as female, following the annotator's convention
            labels = [
                "M" if entry == "M" else ("?" if entry == "?" else "F")
                for entry in entries
            ]
            records.append({"tab": sheet_name, "image": filename, "labels": labels})

    # Close the workbook and return labelled rows
    workbook.close()
    return records

In [ ]:
# Report how many males were marked in each well
def summarise_annotations(records):
    # Count labelled rows and male calls per well
    summary = []
    for tab in sorted({record["tab"] for record in records}):
        male_counts = [
            record["labels"].count("M") for record in records if record["tab"] == tab
        ]
        modal_count, modal_frequency = collections.Counter(male_counts).most_common(1)[0]
        summary.append(
            {
                "tab": tab,
                "rows": len(male_counts),
                "modal_males": modal_count,
                "modal_frequency": modal_frequency,
            }
        )

    # Return one row per well
    return pd.DataFrame(summary)

In [ ]:
# Read the pixels-per-millimetre value from a FlyTracker calibration file
def read_pixels_per_mm(calibration_path):
    # Load the MATLAB structure
    calibration = loadmat(
        calibration_path,
        squeeze_me=True,
        struct_as_record=False,
    )["calib"]

    # Extract the scalar value
    return float(np.ravel(np.asarray(calibration.PPM, dtype=float))[0])

In [ ]:
# Sample a square crop aligned to the fly's body axis
def crop_along_body_axis(image, centre_x, centre_y, orientation, step):
    # Build sampling offsets along the body axis
    offsets = (np.arange(CROP_SIZE, dtype=np.float32) - (CROP_SIZE - 1) / 2) * step

    # Rotate the sampling grid, negating the angle for image coordinates
    unit_x = np.cos(-orientation)
    unit_y = np.sin(-orientation)
    sample_x = centre_x + offsets[None, :] * unit_x + offsets[:, None] * -unit_y
    sample_y = centre_y + offsets[None, :] * unit_y + offsets[:, None] * unit_x

    # Interpolate the image at the sampled positions
    return ndimage.map_coordinates(
        image,
        [sample_y, sample_x],
        order=1,
        mode="nearest",
    )

In [ ]:
# Divide out the slow illumination gradient across the plate
def flatten_illumination(image):
    # Estimate the local background
    background = ndimage.uniform_filter(image, BACKGROUND_WINDOW)

    # Return the ratio of image to background
    return image / np.maximum(background, 1e-6)

In [ ]:
# Convert a flattened crop to unsigned bytes
def quantise_crop(crop):
    # Rescale the working intensity range to 0-255
    scaled = (crop - INTENSITY_FLOOR) / INTENSITY_RANGE * 255

    # Clip and cast
    return np.clip(scaled, 0, 255).astype(np.uint8)

In [ ]:
# Index the usable stills in a session folder by frame number
def index_stills(session_directory):
    # Keep only files large enough to be complete
    stills = {}
    paths = list(session_directory.glob("frame_*.png"))
    paths.extend((session_directory / "photos").glob("frame_*.png"))
    for path in sorted(paths):
        if path.stat().st_size < MINIMUM_FILE_SIZE:
            continue
        frame_number = int(re.search(r"frame_(\d+)", path.name).group(1))
        stills[frame_number] = path

    # Fail early if the folder is empty
    if not stills:
        raise ValueError(f"No usable stills found in {session_directory}")

    # Return the frame-number index
    return stills

In [ ]:
# Extract every labelled fly from one well
def extract_well(records, session_directory, well_number):
    # Locate the FlyTracker output for this well
    matches = sorted(session_directory.rglob(f"*_arena_{well_number}-trackfeat.csv"))
    matches = [path for path in matches if path.is_dir()]
    if len(matches) != 1:
        raise ValueError(f"Expected one trackfeat folder for well {well_number}, found {len(matches)}")
    tracker_directory = matches[0]

    # Load the five per-fly tables
    tracks = [
        pd.read_csv(tracker_directory / f"fly{index}.csv") for index in range(1, 6)
    ]

    # Compute the sampling step that normalises physical scale
    pixels_per_mm = read_pixels_per_mm(tracker_directory / "calibration.mat")
    scale_factor = STILL_WIDTH / PANORAMA_WIDTH
    step = pixels_per_mm / REFERENCE_PPM * scale_factor * FIELD_OF_VIEW

    # Index the stills and find the first frame number
    stills = index_stills(session_directory)
    first_frame = min(stills)
    origin_x, origin_y = WELL_ORIGINS[well_number]

    # Extract one crop per labelled fly
    crops = []
    labels = []
    sources = []
    for record in records:
        # Map the annotation filename to a tracker row
        frame_number = int(re.search(r"_(\d+)\.jpg", record["image"]).group(1))
        if frame_number not in stills:
            continue
        tracker_row = frame_number - first_frame + FRAME_OFFSET

        # Load and flatten the still
        still = np.asarray(Image.open(stills[frame_number]).convert("L"), dtype=np.float32)
        flattened = flatten_illumination(still)

        # Crop each fly that carries a definite label
        for fly_index, label in enumerate(record["labels"]):
            if label not in ("M", "F"):
                continue

            track = tracks[fly_index]
            if tracker_row < 0 or tracker_row >= len(track):
                continue
            if not np.isfinite(track["pos x"].iloc[tracker_row]):
                continue

            centre_x = (track["pos x"].iloc[tracker_row] + origin_x) * scale_factor
            centre_y = (track["pos y"].iloc[tracker_row] + origin_y) * scale_factor
            orientation = float(track["ori"].iloc[tracker_row])

            crop = crop_along_body_axis(
                image=flattened,
                centre_x=centre_x,
                centre_y=centre_y,
                orientation=orientation,
                step=step,
            )
            crops.append(quantise_crop(crop))
            labels.append(1 if label == "M" else 0)
            sources.append((record["tab"], well_number, frame_number, fly_index + 1))

    # Return the crops, labels, and provenance for this well
    return crops, labels, sources

## Run

In [ ]:
# Read and summarise the annotations
annotation_records = read_annotations(workbook_path=WORKBOOK_PATH)
if not annotation_records:
    raise ValueError(f"No annotated rows found in {WORKBOOK_PATH}")
annotation_summary = summarise_annotations(records=annotation_records)

print(f"Annotated rows: {len(annotation_records)}")
print()
print(annotation_summary.to_string(index=False))

In [ ]:
# Group the annotations by well
records_by_tab = collections.defaultdict(list)
for record in annotation_records:
    records_by_tab[record["tab"]].append(record)

# Extract crops well by well
all_crops = []
all_labels = []
all_sources = []
for tab in sorted(records_by_tab):
    session_directory = SESSION_ROOT / SESSION_FOLDERS[tab[:4]]
    well_number = int(tab.split("_a")[1])

    crops, labels, sources = extract_well(
        records=records_by_tab[tab],
        session_directory=session_directory,
        well_number=well_number,
    )

    all_crops.extend(crops)
    all_labels.extend(labels)
    all_sources.extend(sources)

    print(f"{tab} | {len(records_by_tab[tab]):3d} rows | {len(crops):4d} crops")

In [ ]:
# Validate the extracted dataset
if not all_crops:
    raise ValueError("No crops extracted; check annotation frame names and session inputs")

# Stack the results into arrays
crop_array = np.asarray(all_crops, dtype=np.uint8)
label_array = np.asarray(all_labels, dtype=np.float32)
source_array = np.asarray(all_sources, dtype=object)

# Save the dataset
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUTPUT_PATH,
    crops=crop_array,
    y=label_array,
    meta=source_array,
)

print(f"Crops: {crop_array.shape}")
print(f"Male: {int(label_array.sum())} | Female: {int((1 - label_array).sum())}")
print(f"Saved to {OUTPUT_PATH}")